This is hjm-pca based on meuller paper but with only 11 maturities and not full 180 (360 for our case) monthly maturities.

In [ ]:
"""
HJM-PCA Yield Curve Simulator — 11 Maturities (No Interpolation)
Based on Mueller (2022), adapted for FRED US Treasury data.

Pipeline:
  1. Load FRED yields (11 maturities)
  2. Bootstrap forward rates per observation
  3. PCA on weekly forward rate changes → V matrix (11×3)
  4. Compute HJM no-arbitrage drift A from V
  5. Simulate 10,000 scenarios × 60 monthly steps
  6. Convert forward rates directly to yields (no discount factor step)
  7. Save terminal yield curves + full paths
"""

import numpy as np
import pandas as pd

# ── Configuration ──────────────────────────────────────────────────────────
MATURITIES  = np.array([1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])  # years
MAT_LABELS  = ['1M','3M','6M','1Y','2Y','3Y','5Y','7Y','10Y','20Y','30Y']
DTAU        = np.diff(np.concatenate([[0], MATURITIES]))  # interval widths ∆τ_k

YIELD_COLS  = ['Yield_1M','Yield_3M','Yield_6M','Yield_1Y','Yield_2Y',
               'Yield_3Y','Yield_5Y','Yield_7Y','Yield_10Y','Yield_20Y','Yield_30Y']

N_PC        = 3            # principal components (level, slope, curvature)
N_SIM       = 500          # number of scenarios
T_HORIZON   = 5.0          # years
dt          = 1 / 12       # monthly time step
n_steps     = int(T_HORIZON / dt)  # 60 steps

CALIB_END   = '2020-12-31' # calibration cutoff
CSV_PATH    = 'raw_data.csv'
SEED        = 42


# ── Step 0: Load data ──────────────────────────────────────────────────────
df = pd.read_csv(CSV_PATH, parse_dates=['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)
df_valid = df.dropna(subset=YIELD_COLS).copy()

# Calibration sample
df_calib = df_valid[df_valid['DATE'] <= CALIB_END].copy()
yields_calib = df_calib[YIELD_COLS].values / 100.0  # convert % → decimal

# Starting yield curve: last available observation
start_yields = df_valid[YIELD_COLS].iloc[-1].values / 100.0
start_date   = df_valid['DATE'].iloc[-1].date() # 2026-02-20
print(f"Calibration: {df_calib['DATE'].iloc[0].date()} → {df_calib['DATE'].iloc[-1].date()} "
      f"({len(yields_calib):,} obs)")
print(f"Starting curve date: {start_date}")


# ── Step 1: Bootstrap forward rates from zero yields ──────────────────────
#
# Piecewise-constant forward rate for interval [T_{k-1}, T_k]:
#   f_k = (T_k * R_k - T_{k-1} * R_{k-1}) / (T_k - T_{k-1})
#
# For k=0: f_0 = R_0  (first yield = first forward rate directly)
#
def yields_to_forwards(y_vec):
    """Convert zero yields (length 11) to forward rates (length 11)."""
    f = np.empty(len(MATURITIES))
    f[0] = y_vec[0]
    for k in range(1, len(MATURITIES)):
        f[k] = (MATURITIES[k] * y_vec[k] - MATURITIES[k-1] * y_vec[k-1]) / DTAU[k]
    return f


# ── Step 6 (forward declaration): Convert forward rates → yields ───────────
#
# Direct formula (weighted average, no discount factor step needed):
#   R(T_k) = (1 / T_k) * sum_{j=1}^{k} f_j * ∆τ_j
#
def forwards_to_yields(f_vec):
    """Convert forward rates (length 11) to zero yields (length 11)."""
    return np.cumsum(f_vec * DTAU) / MATURITIES


# Build calibration forward curve matrix — shape (N_calib, 11)
print("Bootstrapping forward curves...", end=" ")
F_calib = np.array([yields_to_forwards(yields_calib[i]) for i in range(len(yields_calib))])
print(f"done. Shape: {F_calib.shape}")


# ── Step 2: Weekly changes & annualised covariance ────────────────────────
#
# Thesis: use weekly ∆F, annualise × 52
# Daily data → take every 5th row as weekly proxy
#
F_weekly  = F_calib[::5]
dF_weekly = np.diff(F_weekly, axis=0)          # shape (N_weeks-1, 11)
Sigma_hat = np.cov(dF_weekly.T) * 52           # annualised, shape (11, 11)
print(f"Weekly changes: {dF_weekly.shape} | Covariance: {Sigma_hat.shape} (annualised ×52)")


# ── Step 3: PCA — eigenvectors of Sigma_hat ───────────────────────────────
eigenvalues_raw, Q = np.linalg.eigh(Sigma_hat)
idx         = np.argsort(eigenvalues_raw)[::-1]   # sort descending
eigenvalues = eigenvalues_raw[idx]
Q           = Q[:, idx]

# Scale by sqrt(eigenvalue) — correct HJM volatility convention
# (thesis writes λ but Figure 3 magnitudes confirm √λ is correct)
V = Q[:, :N_PC] * np.sqrt(eigenvalues[:N_PC])    # shape (11, 3)

total_var = eigenvalues.sum()
print("\nPCA variance explained:")
for k in range(N_PC):
    cum = eigenvalues[:k+1].sum() / total_var * 100
    print(f"  PC{k+1}: {eigenvalues[k]/total_var*100:.1f}%  (cumulative: {cum:.1f}%)")


# ── Step 4: HJM no-arbitrage drift ────────────────────────────────────────
#
# Thesis Eq. (15): A_k = σ_k * ∫_0^{T_k} σ(s)^T ds
#
# Discretised with uneven spacing:
#   A_k = sum_j  V_{k,j} * (sum_{i<=k} V_{i,j} * ∆τ_i)
#
def compute_hjm_drift(V, dtau):
    A = np.zeros(V.shape[0])
    for j in range(V.shape[1]):
        integral = np.cumsum(V[:, j] * dtau)   # ∫_0^{T_k} σ_j(s) ds
        A += V[:, j] * integral
    return A

A = compute_hjm_drift(V, DTAU)
print(f"\nDrift A range: [{A.min()*1e4:.2f}, {A.max()*1e4:.2f}] bps/year")


# ── Step 5: Simulate forward curve paths ──────────────────────────────────
#
# Thesis Eq. (20):
#   F_{t+dt} = F_t + (A + ∂F_t/∂τ) * dt + V @ dW
#
# Three terms:
#   1. HJM drift:    A * dt
#   2. Aging/roll:   (∂F/∂τ) * dt — forward difference with uneven spacing
#   3. Stochastic:   V @ (Z * sqrt(dt)),  Z ~ N(0, I_3)
#
# ∂F_k/∂τ ≈ (F_{k+1} - F_k) / (T_{k+1} - T_k)   for k < 11
#          = ∂F_{10}/∂τ                              for k = 11 (repeat last)
#
dtau_diff = np.diff(MATURITIES)   # T_{k+1} - T_k, length 10

F0 = yields_to_forwards(start_yields)

np.random.seed(SEED)
N = len(MATURITIES)

# Store full yield paths: shape (N_SIM, n_steps+1, 11)
Y_paths = np.zeros((N_SIM, n_steps + 1, N))

# Convert initial forward curve to yields and store
Y_paths[:, 0, :] = forwards_to_yields(F0)

# Run simulation
Ft = np.tile(F0, (N_SIM, 1))   # shape (N_SIM, 11) — current forward curves
sqrt_dt = np.sqrt(dt)

print(f"\nSimulating {N_SIM:,} scenarios × {n_steps} monthly steps...")
for t in range(n_steps):
    # Aging term: ∂F/∂τ with uneven spacing (forward difference)
    dFdtau          = np.zeros_like(Ft)
    dFdtau[:, :-1]  = (Ft[:, 1:] - Ft[:, :-1]) / dtau_diff
    dFdtau[:, -1]   = dFdtau[:, -2]              # repeat last difference

    # Stochastic shock: V @ dW,  dW ~ N(0, dt)
    Z     = np.random.randn(N_SIM, N_PC)
    shock = (Z * sqrt_dt) @ V.T                  # shape (N_SIM, 11)

    # Update forward curves
    Ft = Ft + (A + dFdtau) * dt + shock

    # Convert directly to yields and store (no discount factor step)
    Y_paths[:, t + 1, :] = (np.cumsum(Ft * DTAU, axis=1) / MATURITIES)

Y_paths *= 100   # decimal → percent
print("Simulation complete.")


# ── Output ─────────────────────────────────────────────────────────────────
#
# Long format: one row per (scenario, month)
# Columns: Scenario_ID, Month, Y_1M, Y_3M, ..., Y_30Y
# Ordered by Scenario_ID first, then Month (1..60) within each scenario
# t=0 (initial curve) is excluded — only the 60 simulated timesteps
#
OUT_COLS = ['Y_1M','Y_3M','Y_6M','Y_1Y','Y_2Y','Y_3Y','Y_5Y','Y_7Y','Y_10Y','Y_20Y','Y_30Y']

# Y_paths shape: (N_SIM, n_steps+1, 11)
# Drop t=0 (index 0), keep months 1..60 (indices 1..60)
Y_sim = Y_paths[:, 1:, :]   # shape (500, 60, 11)

# Build long-format dataframe
scenario_ids = np.repeat(np.arange(1, N_SIM + 1), n_steps)   # 1,1,...,1, 2,2,...,2, ...
months       = np.tile(np.arange(1, n_steps + 1), N_SIM)      # 1..60, 1..60, ...
yields_flat  = Y_sim.reshape(-1, 11)                           # (30000, 11)

df_out = pd.DataFrame(yields_flat, columns=OUT_COLS)
df_out.insert(0, 'Scenario_ID', scenario_ids)
df_out.insert(1, 'Month', months)

df_out.to_csv('hjm_pca_scenarios.csv', index=False)

# Summary statistics on terminal month (month 60)
print(f"\nTerminal yield statistics at month 60 (%):")
Y_terminal   = Y_sim[:, -1, :]
init_yields  = forwards_to_yields(F0) * 100
print(f"{'Maturity':>8}  {'Initial':>8}  {'Mean':>8}  {'Std':>7}  {'P10':>7}  {'P90':>7}")
print("-" * 55)
for i, lbl in enumerate(OUT_COLS):
    v = Y_terminal[:, i]
    print(f"{lbl:>8}  {init_yields[i]:>8.3f}  {v.mean():>8.3f}  "
          f"{v.std():>7.3f}  {np.percentile(v,10):>7.3f}  {np.percentile(v,90):>7.3f}")

print(f"\nSaved: hjm_pca_scenarios.csv")
print(f"  Rows: {len(df_out):,}  ({N_SIM} scenarios × {n_steps} months)")
print(f"  Columns: {list(df_out.columns)}")
print(f"\nFirst 3 rows:")
print(df_out.head(3).to_string(index=False))


Calibration: 2002-01-02 → 2020-12-31 (4,754 obs)
Starting curve date: 2026-02-20
Bootstrapping forward curves... done. Shape: (4754, 11)
Weekly changes: (950, 11) | Covariance: (11, 11) (annualised ×52)

PCA variance explained:
  PC1: 54.7%  (cumulative: 54.7%)
  PC2: 18.0%  (cumulative: 72.7%)
  PC3: 10.1%  (cumulative: 82.9%)

Drift A range: [0.06, 17.79] bps/year

Simulating 500 scenarios × 60 monthly steps...
Simulation complete.


/var/folders/x3/05_l03tx04300w71f1fld59c0000gn/T/ipykernel_20542/4083368005.py:171: RuntimeWarning: divide by zero encountered in matmul
  shock = (Z * sqrt_dt) @ V.T                  # shape (N_SIM, 11)
/var/folders/x3/05_l03tx04300w71f1fld59c0000gn/T/ipykernel_20542/4083368005.py:171: RuntimeWarning: overflow encountered in matmul
  shock = (Z * sqrt_dt) @ V.T                  # shape (N_SIM, 11)
/var/folders/x3/05_l03tx04300w71f1fld59c0000gn/T/ipykernel_20542/4083368005.py:171: RuntimeWarning: invalid value encountered in matmul
  shock = (Z * sqrt_dt) @ V.T                  # shape (N_SIM, 11)



Terminal yield statistics at month 60 (%):
Maturity   Initial      Mean      Std      P10      P90
-------------------------------------------------------
    Y_1M     3.720     4.045    1.934    1.472    6.615
    Y_3M     3.690     4.065    1.919    1.510    6.597
    Y_6M     3.610     4.096    1.916    1.509    6.625
    Y_1Y     3.510     4.157    1.983    1.495    6.761
    Y_2Y     3.480     4.282    2.095    1.494    6.994
    Y_3Y     3.500     4.387    2.143    1.541    7.188
    Y_5Y     3.650     4.600    2.176    1.728    7.393
    Y_7Y     3.850     4.756    2.165    1.786    7.467
   Y_10Y     4.080     4.939    2.110    2.103    7.586
   Y_20Y     4.660     5.400    1.964    2.730    7.869
   Y_30Y     4.720     5.489    1.858    2.998    7.812

Saved: hjm_pca_scenarios.csv
  Rows: 30,000  (500 scenarios × 60 months)
  Columns: ['Scenario_ID', 'Month', 'Y_1M', 'Y_3M', 'Y_6M', 'Y_1Y', 'Y_2Y', 'Y_3Y', 'Y_5Y', 'Y_7Y', 'Y_10Y', 'Y_20Y', 'Y_30Y']

First 3 rows:
 Scenario_ID